In [ ]:
%py
spark.catalog.setCurrentCatalog("purgo_databricks")

# PySpark script for ETL transformation of Supplier Invoice Oracle Financials fact table
# Purpose: Transform and load supplier invoice data from Oracle Financials into the fact_wf_supplier_invoice_orafin table
# Author: Giang Nguyen
# Date: 2025-10-27
# Description: This script reads required tables from Unity Catalog, applies business logic and joins, enforces schema and data types, handles data quality and error scenarios, and writes the output to the target Delta table partitioned by fiscal year.

# -- Required PySpark imports
from pyspark.sql import DataFrame  
from pyspark.sql import functions as F  
from pyspark.sql.types import (  
    StructType, StructField, StringType, IntegerType, DoubleType, FloatType, ShortType, LongType, DateType, TimestampType
)
from pyspark.sql.window import Window  
from pyspark.sql.utils import AnalysisException  

# -- Helper function: Validate DataFrame schema against expected schema (excluding nullable property)
def validate_schema(df: DataFrame, expected_schema: StructType, table_name: str):
    """
    Validates that DataFrame columns and types match the expected schema (excluding nullable).
    Args:
        df (DataFrame): DataFrame to validate.
        expected_schema (StructType): Expected schema.
        table_name (str): Table name for error reporting.
    Returns:
        None. Raises AssertionError if mismatch.
    """
    df_fields = {f.name: f.dataType for f in df.schema.fields}
    schema_fields = {f.name: f.dataType for f in expected_schema.fields}
    for col, dtype in schema_fields.items():
        assert col in df_fields, f"Missing required field: {col} in {table_name}"
        assert type(df_fields[col]) == type(dtype), f"Type mismatch for {col}: expected {dtype}, got {df_fields[col]} in {table_name}"

# -- Helper function: Deduplicate invoice records by latest snapshot_captured_date per invoice_id
def deduplicate_invoice(df: DataFrame) -> DataFrame:
    """
    Deduplicates invoice records, keeping only the latest snapshot_captured_date per invoice_id.
    Args:
        df (DataFrame): DataFrame with invoice records.
    Returns:
        DataFrame: Deduplicated DataFrame.
    """
    window = Window.partitionBy("invoice_id").orderBy(F.col("snapshot_captured_date").desc())
    return df.withColumn("row_num", F.row_number().over(window)).filter(F.col("row_num") == 1).drop("row_num")

# -- Helper function: Deduplicate expense distribution records by latest xla_manual_override_flag per group
def deduplicate_expense(df: DataFrame) -> DataFrame:
    """
    Deduplicates expense distribution records, keeping only the latest xla_manual_override_flag per group.
    Args:
        df (DataFrame): DataFrame with expense distribution records.
    Returns:
        DataFrame: Deduplicated DataFrame.
    """
    window = Window.partitionBy(
        "invoice_distribution_id", "gl_balancing_segment", "cost_center_segment", "gl_segment1",
        "invoice_id", "distribution_line_number", "invoice_line_number", "invoice_accounting_date", "transaction_amount"
    ).orderBy(F.col("xla_manual_override_flag").desc())
    return df.withColumn("row_num", F.row_number().over(window)).filter(F.col("row_num") == 1).drop("row_num")

# -- Helper function: Load table from Unity Catalog with error handling
def load_uc_table(table_full_name: str) -> DataFrame:
    """
    Loads a table from Unity Catalog with error handling.
    Args:
        table_full_name (str): Full table name (schema.table).
    Returns:
        DataFrame: Loaded DataFrame.
    """
    try:
        return spark.table(table_full_name)
    except Exception as e:
        raise AnalysisException(f"Input table not found: {table_full_name}. Error: {str(e)}")

# -- Helper function: Cast columns to expected types and enforce column order
def enforce_schema(df: DataFrame, expected_schema: StructType) -> DataFrame:
    """
    Casts DataFrame columns to expected types and enforces column order.
    Args:
        df (DataFrame): DataFrame to cast.
        expected_schema (StructType): Expected schema.
    Returns:
        DataFrame: DataFrame with enforced schema.
    """
    select_expr = []
    for field in expected_schema.fields:
        select_expr.append(F.col(field.name).cast(field.dataType).alias(field.name))
    return df.select(*select_expr)

# -- Read input parameters from widgets
target_table_path = dbutils.widgets.get("target_table_path")
partition_key = dbutils.widgets.get("partition")
table_format = dbutils.widgets.get("table_format")
compression = dbutils.widgets.get("compression")
table_name = dbutils.widgets.get("table_name")
unity_catalog = dbutils.widgets.get("unity_catalog")
environment = dbutils.widgets.get("environment")
project = dbutils.widgets.get("project")
load_type = dbutils.widgets.get("load_type")
view_unity_catalog_name = dbutils.widgets.get("view_unity_catalog_name")
raw_unity_catalog = dbutils.widgets.get("raw_unity_catalog")
raw_unity_catalog_hist = dbutils.widgets.get("raw_unity_catalog_hist")
config_unity_catalog = dbutils.widgets.get("config_unity_catalog")
edp_lkp_unity_catalog = dbutils.widgets.get("edp_lkp_unity_catalog")
dims_unity_catalog = dbutils.widgets.get("dims_unity_catalog")
unity_path = f"{unity_catalog}.{table_name}"

# -- Load required tables from Unity Catalog
df_aging = load_uc_table(f"{raw_unity_catalog}.dw_ap_sla_aging_invoice_ca")
df_expense = load_uc_table(f"{raw_unity_catalog}.dw_ap_sla_expense_dist_cf")
df_party = load_uc_table(f"{raw_unity_catalog}.dw_party_d")
df_supplier_site = load_uc_table(f"{raw_unity_catalog}.dw_supplier_site_d")
df_internal_org = load_uc_table(f"{raw_unity_catalog}.dw_internal_org_d_tl")
df_terms = load_uc_table(f"{raw_unity_catalog}.dw_ap_terms_d_tl")
df_natural_account = load_uc_table(f"{raw_unity_catalog}.dw_natural_account_d")
df_payments = load_uc_table(f"{raw_unity_catalog}.dw_ap_sla_payments_cf")
df_gl_segment = load_uc_table(f"{raw_unity_catalog}.dw_gl_segment_d_tl")
df_gl_code_comb = load_uc_table(f"{raw_unity_catalog}.dw_gl_code_combination_d")
df_edp_lkup = load_uc_table(f"{edp_lkp_unity_catalog}.edp_lkup")
df_company = load_uc_table(f"{dims_unity_catalog}.dim_wf_company")

# -- Deduplicate dw_ap_sla_aging_invoice_ca by latest snapshot_captured_date per invoice_id
df_aging_dedup = deduplicate_invoice(df_aging)

# -- Deduplicate dw_ap_sla_expense_dist_cf by latest xla_manual_override_flag per group
df_expense_dedup = deduplicate_expense(df_expense)

# -- Deduplicate dw_ap_sla_payments_cf by check_void_date
df_payments_valid = df_payments.filter(F.col("check_void_date") == "1901-01-01T00:00:00.000+00:00") \
    .groupBy("invoice_id", "invoice_distribution_id", "check_date").agg(F.first("check_void_date").alias("check_void_date"))

# -- Exclusion lists
excluded_supplier_numbers = ["00032238", "00032239", "00080463", "90000147"]
excluded_concat_segments = [
    "4009999110011000000000000000","4009999111011104000000000000","4009999111011105000000000000",
    "4009999111011106000000000000","4009999129012946000000000000","4009999138013800000000000000",
    "4009999172017211000000000000","4009999200020004000000000000","4009999240024042000000000000",
    "4009999240024047000000000000","7009801138013800000000000000","7009801210021031000000000000",
    "7009801210021079000000000000"
]

# -- Prepare lookup DataFrames for GL code combination exclusions
if "concat_segments" in df_gl_code_comb.columns:
    df_gl_code_comb_excl = df_gl_code_comb.withColumn("concat_segments_nodot", F.regexp_replace(F.col("concat_segments"), "\.", ""))
    df_gl_code_comb_excl = df_gl_code_comb_excl.filter(~F.col("concat_segments_nodot").isin(excluded_concat_segments))
else:
    df_gl_code_comb_excl = df_gl_code_comb

# -- Prepare supplier exclusion
if "supplier_number" in df_party.columns:
    df_party_excl = df_party.filter(~F.col("supplier_number").isin(excluded_supplier_numbers))
else:
    df_party_excl = df_party

# -- Prepare supplier site DataFrame with required columns
supplier_site_cols = [c for c in ["supplier_site_id", "PAYMENT_TERMS_ID", "address1", "address2", "city", "state", "country"] if c in df_supplier_site.columns]
df_supplier_site_sel = df_supplier_site.select(*supplier_site_cols)

# -- Join logic for main fact table
# -- CTE: dw_ap_sla_aging_invoice_ca_vw
df_aging_vw = df_aging_dedup

# -- CTE: FAW (main transformation)
# -- Join all required tables
join_exprs = [
    (df_expense_dedup, ["invoice_id"], "left"),
    (df_party_excl, [F.col("dasaic.supplier_party_id") == F.col("dpd.party_id")], "left"),
    (df_supplier_site_sel, ["supplier_site_id"], "left"),
    (df_internal_org, [F.col("dasaic.payables_bu_id") == F.col("diodt.organization_id")], "left"),
    (df_terms, ["PAYMENT_TERMS_ID"], "left"),
    (df_natural_account, ["natural_account_segment"], "left"),
    (df_payments_valid, ["invoice_distribution_id", "invoice_id"], "left"),
    (df_gl_segment.alias("dgsdt_cc"), [(F.col("dgsdt_cc.gl_segment_code") == F.col("dasedc.cost_center_segment")) & (F.col("dgsdt_cc.gl_segment_valueset_code") == F.lit("Center FS_CCG_COA"))], "left"),
    (df_gl_segment.alias("dgsdt_cc_sla"), [(F.col("dgsdt_cc_sla.gl_segment_code") == F.col("dasaic.cost_center_segment")) & (F.col("dgsdt_cc_sla.gl_segment_valueset_code") == F.lit("Center FS_CCG_COA"))], "left"),
    (df_gl_segment.alias("dgsdt_cd"), [(F.col("dgsdt_cd.gl_segment_code") == F.col("dasedc.gl_balancing_segment")) & (F.col("dgsdt_cd.gl_segment_valueset_code") == F.lit("Bal Entity FS_CCG_COA"))], "left"),
    (df_gl_segment.alias("dgsdt_cd_sla"), [(F.col("dgsdt_cd_sla.gl_segment_code") == F.col("dasaic.gl_balancing_segment")) & (F.col("dgsdt_cd_sla.gl_segment_valueset_code") == F.lit("Bal Entity FS_CCG_COA"))], "left"),
    (df_gl_code_comb_excl.alias("dgccd"), [F.col("dasedc.gl_code_combination_id") == F.col("dgccd.code_combination_id")], "left"),
    (df_gl_code_comb_excl.alias("dgccd_sla"), [F.col("dasaic.gl_code_combination_id") == F.col("dgccd_sla.code_combination_id")], "left"),
    (df_gl_segment.alias("dgsdt_acct"), [(F.col("dgsdt_acct.gl_segment_code") == F.col("dasedc.natural_account_segment")) & (F.col("dgsdt_acct.gl_segment_valueset_code") == F.lit("Account FS_CCG_COA"))], "left"),
    (df_gl_segment.alias("dgsdt_acct_sla"), [(F.col("dgsdt_acct_sla.gl_segment_code") == F.col("dasaic.natural_account_segment")) & (F.col("dgsdt_acct_sla.gl_segment_valueset_code") == F.lit("Account FS_CCG_COA"))], "left"),
    (df_gl_segment.alias("dgsdt_subacct"), [(F.col("dgsdt_subacct.gl_segment_code") == F.col("dasedc.gl_segment1")) & (F.col("dgsdt_subacct.gl_segment_valueset_code") == F.lit("Sub Account FS_CCG_COA"))], "left"),
    (df_gl_segment.alias("dgsdt_subacct_sla"), [(F.col("dgsdt_subacct_sla.gl_segment_code") == F.col("dasaic.gl_segment1")) & (F.col("dgsdt_subacct_sla.gl_segment_valueset_code") == F.lit("Sub Account FS_CCG_COA"))], "left"),
    (df_edp_lkup.alias("edp_lkup"), [(F.col("edp_lkup.lkup_key_02") == F.coalesce(F.col("dasedc.cost_center_segment"), F.col("dasaic.cost_center_segment"))) & (F.col("edp_lkup.lkup_typ_nm") == F.lit("TFS_CC_TO_DIV")) & (F.lower(F.col("edp_lkup.lkup_key_01")) == F.lit("usorafin"))], "left"),
    (df_edp_lkup.alias("edp_lkup_div"), [
        (F.col("edp_lkup_div.lkup_key_02") == F.coalesce(F.col("dasedc.gl_balancing_segment"), F.col("dasaic.gl_balancing_segment"))) &
        (F.col("edp_lkup_div.lkup_key_03") == F.coalesce(F.col("dasedc.cost_center_segment"), F.col("dasaic.cost_center_segment"))) &
        (F.col("edp_lkup_div.lkup_key_04") == F.concat_ws("-", F.coalesce(F.col("dasedc.natural_account_segment"), F.col("dasaic.natural_account_segment")), F.coalesce(F.col("dasedc.gl_segment1"), F.col("dasaic.gl_segment1")))) &
        (F.col("edp_lkup_div.lkup_typ_nm") == F.lit("TFS_COCD_CCNTR_GLACCT_TO_DIVCD")) &
        (F.upper(F.col("edp_lkup_div.lkup_key_01")) == F.lit("USORAFIN"))
    ], "left"),
    (df_edp_lkup.alias("edp_lkup_div_1"), [
        (F.col("edp_lkup_div_1.lkup_key_02") == F.coalesce(F.col("dasedc.gl_balancing_segment"), F.col("dasaic.gl_balancing_segment"))) &
        (F.col("edp_lkup_div_1.lkup_key_03") == F.coalesce(F.col("dasedc.cost_center_segment"), F.col("dasaic.cost_center_segment"))) &
        (F.col("edp_lkup_div_1.lkup_key_04").isNull()) &
        (F.col("edp_lkup_div_1.lkup_typ_nm") == F.lit("TFS_COCD_CCNTR_GLACCT_TO_DIVCD")) &
        (F.upper(F.col("edp_lkup_div_1.lkup_key_01")) == F.lit("USORAFIN"))
    ], "left"),
    (df_edp_lkup.alias("edp_lkup_payment"), [
        (F.upper(F.col("edp_lkup_payment.lkup_key_02")) == F.upper(F.col("datdt.payment_term_name"))) &
        (F.col("edp_lkup_payment.lkup_typ_nm") == F.lit("PAYMENT_TERMS_MAPPING")) &
        (F.upper(F.col("edp_lkup_payment.lkup_key_01")) == F.lit("USORAFIN"))
    ], "left"),
    (df_company.alias("comp"), [F.col("comp.co_cd") == F.coalesce(F.col("dasedc.gl_balancing_segment"), F.col("dasaic.gl_balancing_segment"))], "left")
]

df_faw = df_aging_vw.alias("dasaic")
for join_df, join_cond, join_type in join_exprs:
    if isinstance(join_cond, list):
        cond = join_cond[0] if len(join_cond) == 1 else join_cond[0]
        for c in join_cond[1:]:
            cond = cond & c
        df_faw = df_faw.join(join_df, cond, join_type)
    else:
        df_faw = df_faw.join(join_df, join_cond, join_type)

# -- Apply business logic filters
filters = [
    (F.year(F.col("dasaic.invoice_accounting_date")) >= (F.year(F.current_timestamp()) - 3)),
    (~F.col("dasaic.invoice_source_code").isin(["Receivables"])),
    (
        (~((F.col("dasedc.invoice_source_code") == "MAINFRAME") &
           (F.col("dasedc.invoice_description").rlike("^DS|^DR|^PE|^PR|^PS|^RG|^TR"))) |
         F.col("dasedc.invoice_description").isNull())
    ),
    (
        ((F.col("dasedc.natural_account_segment").cast("int").between(4000, 8999)) | F.col("dasedc.natural_account_segment").isNull()) |
        ((F.col("dasedc.cost_center_segment").cast("int").between(9000, 9999)) | F.col("dasedc.cost_center_segment").isNull())
    )
]

if "concat_segments" in df_gl_code_comb_excl.columns:
    filters.append(~F.regexp_replace(F.col("dgccd.concat_segments"), "\.", "").isin(excluded_concat_segments))
    filters.append(~F.regexp_replace(F.col("dgccd_sla.concat_segments"), "\.", "").isin(excluded_concat_segments))
if "supplier_number" in df_party_excl.columns:
    filters.append(~F.col("dpd.supplier_number").isin(excluded_supplier_numbers))

df_faw_filtered = df_faw
for f in filters:
    df_faw_filtered = df_faw_filtered.filter(f)

# -- Select and map output columns, enforcing types and order
output_columns = [
    ("document_type", StringType(), F.lit(None)),
    ("txn_ref_nbr", StringType(), F.lit(None)),
    ("invc_entry_period", StringType(), F.date_format(F.col("dasaic.invoiced_on_date"), "yyyyMM")),
    ("po_nbr", StringType(), F.lit(None)),
    ("po_line_nbr", StringType(), F.lit(None)),
    ("src_sys_cd", StringType(), F.lit("usorafin")),
    ("vchr_nbr", StringType(), F.col("dasaic.invoice_id").cast(StringType())),
    ("vchr_line_nbr", StringType(), F.concat_ws("-", F.col("dasedc.invoice_line_number"), F.col("dasedc.distribution_line_number")).cast(StringType())),
    ("fscl_yr_nbr", StringType(), F.date_format(F.col("dasedc.invoice_accounting_date"), "yyyy")),
    ("vchr_type_cd", StringType(), F.col("dasaic.invoice_type_code")),
    ("vchr_status", StringType(), F.lit(None)),
    ("item_nbr", StringType(), F.lit(None)),
    ("item_desc", StringType(), F.lit(None)),
    ("thermo_item_nbr", StringType(), F.lit(None)),
    ("supplier_cd", StringType(), F.concat_ws("_", F.coalesce(F.col("dpd.supplier_number"), F.lit("0")), F.col("dssd.supplier_site_id"))),
    ("supplier_name", StringType(), F.col("dpd.party_name")),
    ("supplier_type_cd", StringType(), F.lit(None)),
    ("buyer_cd", StringType(), F.lit(None)),
    ("document_desc", StringType(), F.lit(None)),
    ("invc_txn_type", StringType(), F.lit(None)),
    ("buyer_nm", StringType(), F.lit(None)),
    ("co_cd", StringType(), F.coalesce(F.col("dasedc.gl_balancing_segment"), F.col("dasaic.gl_balancing_segment"))),
    ("co_name", StringType(), F.col("comp.co_nm")),
    ("hfm_entity", StringType(), F.col("edp_lkup.lkup_val_03")),
    ("business_unit", StringType(), F.col("diodt.organization_name")),
    ("lcr_flag", StringType(), F.lit(None)),
    ("lcr_region", StringType(), F.lit(None)),
    ("vomi_flag", StringType(), F.lit(None)),
    ("payment_compliance_flg", StringType(), F.lit(None)),
    ("po_curncy_cd", StringType(), F.col("dasaic.transaction_currency_code")),
    ("co_curncy_cd", StringType(), F.col("dasaic.ledger_currency_code")),
    ("post_yr_mth_nbr", StringType(), F.date_format(F.col("dasedc.invoice_accounting_date"), "yyyyMM")),
    ("invc_entry_dt", StringType(), F.date_format(F.col("dasaic.invoiced_on_date"), "yyyyMMdd")),
    ("paymt_due_dt", StringType(), F.date_format(F.col("dasaic.invoice_schedule_due_date"), "yyyyMMdd")),
    ("suplr_invc_dt", StringType(), F.date_format(F.col("dasaic.invoice_accounting_date"), "yyyyMMdd")),
    ("aprval_dt", StringType(), F.lit(None)),
    ("txn_orig_id", StringType(), F.lit(None)),
    ("suplr_invc_nbr", StringType(), F.col("dasaic.invoice_number")),
    ("invc_apprv_id", StringType(), F.lit(None)),
    ("unit_prc", DoubleType(), F.col("dasedc.transaction_amount").cast(DoubleType())),
    ("invc_qty", DoubleType(), F.lit(1.0)),
    ("base_qty", DoubleType(), F.lit(None)),
    ("invc_txn_amt", DoubleType(), F.col("dasedc.transaction_amount").cast(DoubleType())),
    ("invc_co_amt", DoubleType(), F.col("dasedc.transaction_amount").cast(DoubleType())),
    ("invc_txn_pmar_amt", DoubleType(), F.lit(None)),
    ("invc_co_pmar_amt", DoubleType(), F.lit(0.0)),
    ("unit_prc_pmar_amt", DoubleType(), F.lit(0.0)),
    ("txn_curncy_mth_rt", DoubleType(), F.lit(0.0)),
    ("co_curncy_mth_rt", DoubleType(), F.lit(0.0)),
    ("uom_conv_factor", DoubleType(), F.lit(None)),
    ("invc_uom_cd", StringType(), F.lit(None)),
    ("base_uom_cd", StringType(), F.lit(None)),
    ("profit_cntr", StringType(), F.lit(None)),
    ("div_cd", StringType(),
        F.when(
            F.coalesce(F.col("edp_lkup_div.lkup_val_01"), F.col("edp_lkup_div_1.lkup_val_01")).isNull(),
            F.when(F.coalesce(F.col("dasedc.gl_balancing_segment"), F.col("dasaic.gl_balancing_segment")) == "400", F.lit("CCG Group"))
             .when(F.coalesce(F.col("dasedc.gl_balancing_segment"), F.col("dasaic.gl_balancing_segment")) == "700", F.lit("Corporate"))
        ).otherwise(F.coalesce(F.col("edp_lkup_div.lkup_val_01"), F.col("edp_lkup_div_1.lkup_val_01")))
    ),
    ("site_cd", StringType(), F.lit(None)),
    ("site_name", StringType(), F.lit(None)),
    ("reporting_site", StringType(), F.lit(None)),
    ("warehouse", StringType(), F.lit(None)),
    ("warehouse_nm", StringType(), F.lit(None)),
    ("unit", StringType(), F.lit(None)),
    ("nature", StringType(), F.lit(None)),
    ("inv_flg", StringType(), F.lit(None)),
    ("inv_flg_text", StringType(), F.lit(None)),
    ("spend_type_cd", StringType(), F.lit("Indirect")),
    ("po_paymt_terms_cd", StringType(), F.lit(None)),
    ("po_paymt_terms_desc", StringType(), F.lit(None)),
    ("suplr_paymt_terms_cd", StringType(), F.col("datdt.payment_term_name")),
    ("suplr_paymt_terms_desc", StringType(), F.coalesce(F.col("datdt.payment_term_description"), F.col("edp_lkup_payment.lkup_val_01"))),
    ("fk_orig", StringType(), F.lit(None)),
    ("floor_stock_cd", StringType(), F.lit(None)),
    ("contract_flag", StringType(), F.lit(None)),
    ("contract_type", StringType(), F.lit(None)),
    ("contract_start_date", StringType(), F.lit(None)),
    ("contract_end_date", StringType(), F.lit(None)),
    ("erp_commondity_cd", StringType(), F.lit(None)),
    ("erp_commondity_nm", StringType(), F.lit(None)),
    ("sec_supp_cd", StringType(), F.lit(None)),
    ("part_rev_no", StringType(), F.lit(None)),
    ("cost_centre_cd", StringType(), F.coalesce(F.col("dasedc.cost_center_segment"), F.col("dasaic.cost_center_segment"))),
    ("cost_centre_nm", StringType(), F.coalesce(F.col("dgsdt_cc.gl_segment_description"), F.col("dgsdt_cc_sla.gl_segment_description"))),
    ("vendor_mat_no", StringType(), F.lit(None)),
    ("gl_acct_id", StringType(), F.concat_ws("-", F.coalesce(F.col("dasedc.gl_balancing_segment"), F.col("dasaic.gl_balancing_segment")), F.coalesce(F.col("dasedc.natural_account_segment"), F.col("dasaic.natural_account_segment")), F.coalesce(F.col("dasedc.gl_segment1"), F.col("dasaic.gl_segment1")))),
    ("gl_acct_nm", StringType(), F.concat_ws("-", F.coalesce(F.col("dgsdt_acct.gl_segment_description"), F.col("dgsdt_acct_sla.gl_segment_description")), F.coalesce(F.col("dgsdt_subacct.gl_segment_description"), F.col("dgsdt_subacct_sla.gl_segment_description")))),
    ("pass_through_field", StringType(), F.lit(None)),
    ("pass_through_line", StringType(), F.lit(None)),
    ("inv_line_desc", StringType(), F.col("dasedc.invoice_description")),
    ("remit_to_addr_line_1", StringType(), F.col("dssd.address1")),
    ("remit_to_addr_line_2", StringType(), F.col("dssd.address2")),
    ("remit_to_addr_line_3", StringType(), F.lit(None)),
    ("remit_to_addr_line_4", StringType(), F.lit(None)),
    ("remit_to_city_nm", StringType(), F.col("dssd.city")),
    ("remit_to_st_cd", StringType(), F.col("dssd.state")),
    ("remit_to_rgn_cd", StringType(), F.lit(None)),
    ("remit_to_rgn_nm", StringType(), F.lit(None)),
    ("remit_to_cntry_cd", StringType(), F.col("dssd.country")),
    ("remit_to_cntry_nm", StringType(), F.lit(None)),
    ("suplr_nm_src", StringType(), F.lit(None)),
    ("rpt_flex1", StringType(), F.lit(None)),
    ("invc_txn_amt_clsfctn", StringType(), F.lit(None)),
    ("supplier_segment", StringType(), F.lit(None)),
    ("ap_payment_term_cd", StringType(), F.col("datdt.payment_term_name")),
    ("ap_payment_term_desc", StringType(), F.coalesce(F.col("datdt.payment_term_description"), F.col("edp_lkup_payment.lkup_val_01"))),
    ("actual_payment_dt", StringType(), F.date_format(F.col("daspc.check_date"), "yyyy-MM-dd")),
    ("source_country", StringType(), F.lit("NA"))
]

select_expr = [expr.alias(name) for name, dtype, expr in output_columns]
df_output = df_faw_filtered.select(*select_expr)

# -- Define output schema for validation
output_schema = StructType([
    StructField(name, dtype, True) for name, dtype, _ in output_columns
])

# -- Validate output DataFrame schema
validate_schema(df_output, output_schema, table_name)

# -- Write output DataFrame to Delta table with partitioning and compression
df_output.write.format(table_format).mode("overwrite").option("compression", compression).partitionBy(partition_key).save(target_table_path)

# -- Register table in Unity Catalog
df_output.write.format("delta").mode("overwrite").partitionBy(partition_key).saveAsTable(unity_path)

# END OF SCRIPT